<a href="https://colab.research.google.com/github/EsatovinVictoria/ML_Semester5_EsatovinEbenhaezerVictoria/blob/main/Pertemuan_6/Praktikum_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Praktikum 6

In [ ]:
import pandas as pd
import numpy as np
import time
import faiss

from annoy import AnnoyIndex
import hnswlib

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('/content/drive/MyDrive/songs_with_attributes_and_lyrics.csv')

features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

X = df[features].values

print("Jumlah data :", X.shape[0])
print("Jumlah fitur:", X.shape[1])

Jumlah data : 955320
Jumlah fitur: 9


In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Disarankan untuk FAISS dan HNSW
X_scaled = X_scaled.astype(np.float32)

k = 10

In [ ]:
start = time.time()

nn = NearestNeighbors(
    n_neighbors=k,
    algorithm='brute',
    metric='euclidean'
)

nn.fit(X_scaled)

dist_exact, idx_exact = nn.kneighbors(X_scaled)

time_exact = time.time() - start

print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 4445.825 s


In [ ]:
start = time.time()

f = X_scaled.shape[1]

index_annoy = AnnoyIndex(
    f,
    'euclidean'
)

for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)

index_annoy.build(10)

idx_annoy = [
    index_annoy.get_nns_by_vector(v, k)
    for v in X_scaled
]

time_annoy = time.time() - start

print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 66.036 s


Menjalankan HNSW

In [ ]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679796 sha256=7dc4b0ccae8b9cadd72dbe3d369381639071a8b617029106d5ed627dca2998ff
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [ ]:
import time
import hnswlib
import numpy as np

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Ganti path sesuai lokasi file CSV kamu
df = pd.read_csv('/content/drive/MyDrive/songs_with_attributes_and_lyrics.csv')

features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

k = 10

print("Jumlah data :", X_scaled.shape[0])
print("Jumlah fitur:", X_scaled.shape[1])

Jumlah data : 955320
Jumlah fitur: 9


In [ ]:
start = time.time()

p_hnsw = hnswlib.Index(
    space='l2',
    dim=X_scaled.shape[1]
)

p_hnsw.init_index(
    max_elements=X_scaled.shape[0],
    ef_construction=200,
    M=16
)

p_hnsw.add_items(X_scaled)

p_hnsw.set_ef(200)

idx_hnsw, dist_hnsw = p_hnsw.knn_query(
    X_scaled,
    k=k
)

time_hnsw = time.time() - start

print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 382.407 s


Menjalankan FAISS IVF

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.1 MB/s eta 0:00:00


In [ ]:
import faiss
import time

In [ ]:
import faiss
import time

start = time.time()

quantizer = faiss.IndexFlatL2(
    X_scaled.shape[1]
)

index_faiss = faiss.IndexIVFFlat(
    quantizer,
    X_scaled.shape[1],
    100,
    faiss.METRIC_L2
)

index_faiss.train(X_scaled)

index_faiss.add(X_scaled)

index_faiss.nprobe = 10

dist_faiss, idx_faiss = index_faiss.search(
    X_scaled,
    k
)

time_faiss = time.time() - start

print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 1054.819 s


Membandingkan Track Terdekat

| Model | Metode | Waktu Komputasi | Top-5 Neighbor |
|---|---|---:|---|
| Exact NN | Brute Force | 4445.825 s | Tidak tercatat |
| Annoy | Approximate NN | 66.036 s | Tidak tercatat |
| HNSW | Graph-based ANN | 382.407 s | Tidak tercatat |
| FAISS IVF | Inverted File ANN | 1054.819 s | Tidak tercatat |